## Hybrid search with Qdrant

Vector search based on dense embeddings captures the semantics of the data, so you don't have to use the same terms in queries and documents to still be able to find relevant items. However, historically we were also using some other methods which rely on the presence of the keywords. Methods such as Bag-of-words, TFIDF and BM25 are still useful in some cases should be preferred over the dense embeddings.

### Sparse vectors

Surprisingly, keyword-based search is also implemented as vector search, but these vectors are usually sparse. That means the majority of the dimensions of such a vector are just zeros. A non zero value at a particular vector dimension indicates the presence of a term from the dictionary assigned to that position. In other words, in sparse vectors, we have a dictionary in which each word/phrase gets its unique position. Since vectors are sparse, the dictionary can theoretically grow indefinitely, as we can append a new term at the very end.

The fact of using a flexible dictionary, make the sparse vectors excel in exact matches, as they can cover texts that would be sets of random characters for the dense vectors - such as proper names or identifiers. Dense embedding models also have a dictionary, but once the model is trained, extending them is not that easy, and requires fine-tuning of the model. A typical user rarely goes that far.

### BM25

Thera are plenty of different options for creating sparse embeddings, but BM25 is an industry standard, and its most popular form comes from the 90s. It's a statistical model(no neural networks involved), which makes it really fast and lightweight. It's actually a solid baseline in search benchmarks so you should not ignore it.

BM25 stands for Best Matching 25, and it was just the 25 attemp to create a formula that calculates how relevant a particular document is, gicen a query. If you are interested in mathematical background, please check out the Wikipedia page that describes it really well. In general, BM25 is a ranking function that helps search engines determine how relevant a document is to a query by combining two key concepts: **Term Frequency (TF)** and **Inverse Document Frequency (IDF)**.


1. The Term frequency componenent rewards documents that contain the query terms multiple times, but with diminishing returns - so a document with 10 occurences of a word isn't necessarily 10 times better than one with just 01 occurence.

2. The Inverse Document Frequency part boots the importance of rare words while reducing the weight of common words that appear in many documents, since rare terms are typically more informative for distinguishing relevant results.



BM25 also incorporates document length normalization to prevent longer documents from having an unfair advantage simply due to their size. 
In our case, we'll use an implementation available in FastEmbed. Let's start with the basics.

In [1]:
from qdrant_client import QdrantClient 

client = QdrantClient(
    url="http://localhost:6333",
   
)

client.get_collections()



CollectionsResponse(collections=[CollectionDescription(name='zoomcamp-rag')])

### Sparse vector search with BM25

In [2]:
import requests 

docs_url = 'https://github.com/alexeygrigorev/llm-rag-workshop/raw/main/notebooks/documents.json'
docs_response = requests.get(docs_url)
documents_raw = docs_response.json()

We need to create the collection first. Qdrant will handle the IDF calculations, if we configure it to. That's required for BM25, otherwise it won't boost the rare words.

In [3]:
from qdrant_client import models

# Create a collection with specified sparse vector parameters
client.create_collection(
    collection_name = "zoomcamp-sparse", 
    sparse_vectors_config = {
        "bm25": models.SparseVectorParams(
            modifier = models.Modifier.IDF, 
        )
    }
)

True

FastEmbed comes with a BM25 implementation that we can use as any other model

In [5]:
import uuid 


client.upsert(
    collection_name = "zoomcamp-sparse", 
    points = [
        models.PointStruct(
            id = uuid.uuid4().hex, 
            vector = {

                "bm25": models.Document(
                    text = doc["text"], 
                    model = "Qdrant/bm25"
                ), 
            }, 
            payload = {
                "text": doc["text"], 
                "section": doc["section"], 
                "course": course["course"],
            }
        )
        for course in documents_raw
        for doc in course["documents"]
    ]
)

Fetching 18 files: 100%|██████████| 18/18 [00:01<00:00, 14.15it/s]


UpdateResult(operation_id=1, status=<UpdateStatus.COMPLETED: 'completed'>)

### Step 3 : Running sparse vector search with BM25

Right now, our vectors are ready to be searched over. Let's create a helper function.

In [6]:
def search(query:str, limit: int = 1) -> list[models.ScoredPoint]:

    results = client.query_points(
        collection_name = "zoomcamp-sparse", 
        query = models.Document(
            text = query, 
            model = "Qdrant/bm25"
        ), 
        using="bm25", 
        limit=limit, 
        with_payload = True, 
    )
    return results.points

In [8]:
results = search("engineering join")
results

[ScoredPoint(id='4d956767-4b28-44b1-a39e-c78e154dd048', version=1, score=8.705321, payload={'text': "Here’s how you join a in Slack: https://slack.com/help/articles/205239967-Join-a-channel\nClick “All channels” at the top of your left sidebar. If you don't see this option, click “More” to find it.\nBrowse the list of public channels in your workspace, or use the search bar to search by channel name or description.\nSelect a channel from the list to view it.\nClick Join Channel.\nDo we need to provide the GitHub link to only our code corresponding to the homework questions?\nYes. You are required to provide the URL to your repo in order to receive a grade", 'section': 'General course-related questions', 'course': 'machine-learning-zoomcamp'}, vector=None, shard_key=None, order_value=None)]

Sparse vectors can return no results, if none of the keywords from the query were ever used in the documents. No matter if there are some synonyms. Terminology does matter.

In [9]:
results = search("pandas")
print(results[0].payload["text"])

You can use round() function or f-strings
round(number, 4)  - this will round number up to 4 decimal places
print(f'Average mark for the Homework is {avg:.3f}') - using F string
Also there is pandas.Series. round idf you need to round values in the whole Series
Please check the documentation
https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.Series.round.html#pandas.Series.round
Added by Olga Rudakova


Score returned by BM25 are not calculated with cosine similarity, but with BM25 formula. They are not bounded to a specific range, but are virtually unbounded. Let's see how they may look like.

In [10]:
results[0].score

6.0370936

That's an important observation before we start implementing hybrid search.

### Natural language like queries

Let's try again with a random question from our dataset to see how well sparse vector search can work with longer, natural language like queries.

In [11]:
import random
import json
random.seed(42)



course = random.choice(documents_raw)
course_piece = random.choice(course["documents"])
print(json.dumps(course_piece, indent=2))



{
  "text": "I have faced a problem while reading the large parquet file. I tried some workarounds but they were NOT successful with Jupyter.\nThe error message is:\nIndexError: index 311297 is out of bounds for axis 0 with size 131743\nI solved it by performing the homework directly as a python script.\nAdded by Ibraheem Taha (ibraheemtaha91@gmail.com)\nYou can try using the Pyspark library\nAnswered by kamaldeen (kamaldeen32@gmail.com)",
  "section": "Module 1: Introduction",
  "question": "Reading large parquet files"
}


In [12]:
results = search(course_piece["question"])
print(results[0].payload["text"])

I have faced a problem while reading the large parquet file. I tried some workarounds but they were NOT successful with Jupyter.
The error message is:
IndexError: index 311297 is out of bounds for axis 0 with size 131743
I solved it by performing the homework directly as a python script.
Added by Ibraheem Taha (ibraheemtaha91@gmail.com)
You can try using the Pyspark library
Answered by kamaldeen (kamaldeen32@gmail.com)


Step 4 : Qdrant Universal Query API - prefectching

Qdrant's **.query_points** method allows building multi-step search pipelines which can incorporate various methods into a single call. For example, we can retrieve some candidates with dense vector, and then rerank them with sparse each, or use a fast method for initial retrieval and precise, but slow, reranking.

Let's create another collection that will keep both dense and sparse representations. Qdrant named vectors allow us to store multiple representations per point and it proves useful especially when we want to use multiple models in our applications.

In [13]:
# Create the collection with both vector types

client.create_collection(
    collection_name = "zoomcamp-sparse-dense", 
    vectors_config = {
        "jina-small": models.VectorParams(
            size = 512, 
            distance = models.Distance.COSINE
        ), 

    }, 
    sparse_vectors_config = {
        "bm25": models.SparseVectorParams(
            modifier = models.Modifier.IDF, 
        )
    }
)

True

We have to upload all the vectors into the newly created collection.

In [16]:
client.upsert(
    collection_name = "zoomcamp-sparse-dense", 
    points = [
        models.PointStruct(
            id = uuid.uuid4().hex, 
            vector = {
                "jina-small": models.Document(
                    text = doc["text"], 
                    model = "jinaai/jina-embeddings-v2-small-en"
                ), 

                "bm25": models.Document(
                    text = doc["text"], 
                    model = "Qdrant/bm25"
                ), 
            }, 
            payload = {
                "text": doc["text"], 
                "section": doc["section"], 
                "course": course["course"],
            }

            
        )
        for course in documents_raw
            for doc in course["documents"]

    ]
)

UpdateResult(operation_id=1, status=<UpdateStatus.COMPLETED: 'completed'>)

In [17]:
def multi_stage_search(query:str, limit: int = 1) -> list[models.ScoredPoint]:

    # First stage - dense vector search
    dense_results = client.query_points(
        collection_name = "zoomcamp-sparse-dense", 
        prefetch = [
            models.Prefetch(
                query = models.Document(
                    text = query, 
                    model = "jinaai/jina-embeddings-v2-small-en", 
                ), 
                using = "jina-small", 
                # Prefetch ten times more results, then 
                # expected to return, so we can really rerank
                limit=(10 * limit)
            ),
        ], 

        query = models.Document(
            text = query, 
            model = "Qdrant/bm25", 
        ), 
        using = "bm25",
        limit = limit, 
        with_payload = True, 
            
        )

    return dense_results.points 

In [18]:
print(json.dumps(course_piece, indent=2))

{
  "text": "I have faced a problem while reading the large parquet file. I tried some workarounds but they were NOT successful with Jupyter.\nThe error message is:\nIndexError: index 311297 is out of bounds for axis 0 with size 131743\nI solved it by performing the homework directly as a python script.\nAdded by Ibraheem Taha (ibraheemtaha91@gmail.com)\nYou can try using the Pyspark library\nAnswered by kamaldeen (kamaldeen32@gmail.com)",
  "section": "Module 1: Introduction",
  "question": "Reading large parquet files"
}


In [19]:
results = multi_stage_search(course_piece["question"])
print(results[0].payload["text"])


I have faced a problem while reading the large parquet file. I tried some workarounds but they were NOT successful with Jupyter.
The error message is:
IndexError: index 311297 is out of bounds for axis 0 with size 131743
I solved it by performing the homework directly as a python script.
Added by Ibraheem Taha (ibraheemtaha91@gmail.com)
You can try using the Pyspark library
Answered by kamaldeen (kamaldeen32@gmail.com)


## Step 5 : Building Hybrid Search

In real production systems, you don't need to choose just one vector type. You never know what kind of queries your users will send to the system. E-commerce search might be just fine with lexical search on top of sparse vectors, as people will tend to send keywords, but in conversational systems, such as chatbots, natural language questions might be more frequent. Using one model as a retriever and another one as reranker is not the only way of how to  use dense and sparse in a single system.


**Hybrid Search** is a technique for combining results coming from different search methods - for example dense and sparse. There isn't a clear definition of how exactly to implement it, as the main problem is how to mix results coming from methods which are incompatible. Dense and sparse search scores can't be compared directly, so we need another method that will order the final results somehow.

There are two terms important for Hybrid Search : **fusion** and **reranking**

### Fusion 


Fusion is a set of methods which work on the scores/ranking as returned by the individual methods. There are various ways of how to achieve that, but Reciprocal Rank Fusion is the most popular technique. It is based on the rankings of the documents in each methods used, and these rankings are used to calculate the final scores. You will never calculate these scores, as Qdrant has some built-in capabilities that we will use. However, the following example can give you a rough intuition:




In [30]:
def rrf_search(query:str, limit: int = 1) -> list[models.ScoredPoint]:
    """
    Performs hybrid search using Reciprocal Rank Fusion (RRF).
    
    Both dense and sparse searches run in parallel on the full collection,
    then results are merged using RRF.
    """
    results = client.query_points(
        collection_name = "zoomcamp-sparse-dense", 
        # Prefetch: Run multiple searches in parallel
        prefetch = [
            models.Prefetch(
                query = models.Document(
                    text = query, 
                    model = "jinaai/jina-embeddings-v2-small-en", 
                ), 
                using = "jina-small", 
                limit = (10 * limit),  # Get more candidates for better fusion
            ), 
            models.Prefetch(
                query = models.Document(
                    text = query, 
                    model = "Qdrant/bm25", 
                ), 
                using = "bm25", 
                limit = (10 * limit),  # Get more candidates for better fusion
            ), 
        ], 
        # Fusion query: Merge prefetch results using RRF
        query = models.FusionQuery(fusion=models.Fusion.RRF), 
        limit = limit,  # Final number of results to return
        with_payload = True,  
    )
    
    return results.points

In [31]:
results = rrf_search(course_piece["question"])
print(json.dumps(course_piece, indent=2))
print(results[0].payload["text"])

{
  "text": "I have faced a problem while reading the large parquet file. I tried some workarounds but they were NOT successful with Jupyter.\nThe error message is:\nIndexError: index 311297 is out of bounds for axis 0 with size 131743\nI solved it by performing the homework directly as a python script.\nAdded by Ibraheem Taha (ibraheemtaha91@gmail.com)\nYou can try using the Pyspark library\nAnswered by kamaldeen (kamaldeen32@gmail.com)",
  "section": "Module 1: Introduction",
  "question": "Reading large parquet files"
}
I have faced a problem while reading the large parquet file. I tried some workarounds but they were NOT successful with Jupyter.
The error message is:
IndexError: index 311297 is out of bounds for axis 0 with size 131743
I solved it by performing the homework directly as a python script.
Added by Ibraheem Taha (ibraheemtaha91@gmail.com)
You can try using the Pyspark library
Answered by kamaldeen (kamaldeen32@gmail.com)


### RRF vs Reranking: Key Differences

**They are DIFFERENT approaches**, not the same thing!

#### RRF (Reciprocal Rank Fusion) = FUSION

**Process**: Run multiple searches **in parallel** → Merge their results

**Flow Diagram:**
```
Dense Search (full collection) ──┐
                                  ├──> Merge with RRF formula ──> Final ranking
Sparse Search (full collection) ──┘
```

**Key characteristics:**
- ✅ Both searches run **independently** on the **entire dataset**
- ✅ Each search returns its own ranked list
- ✅ Results are **merged** using the RRF formula
- ✅ Documents appearing in multiple lists get boosted
- ✅ **Parallel execution** - searches can run simultaneously

**Example:**
- Dense search returns: [Doc_A, Doc_B, Doc_C] (from entire collection)
- Sparse search returns: [Doc_C, Doc_A, Doc_D] (from entire collection)
- RRF merges: Doc_A (appears in both!) > Doc_C > Doc_B > Doc_D

---

#### Reranking = TWO-STAGE PROCESS

**Process**: Run one search → Filter candidates → Rerank ONLY those candidates

**Flow Diagram:**
```
Dense Search (full collection) ──> Get top 10 candidates ──> BM25 reranks ONLY those 10 ──> Final ranking
```

**Key characteristics:**
- ✅ First search finds candidates from **entire collection**
- ✅ Second search operates **ONLY on the filtered subset**
- ✅ Sequential process - must wait for stage 1 to complete
- ✅ More efficient - only reranks a small set
- ✅ Leverages synergy: one method filters, another ranks precisely

**Example (from your `multi_stage_search` function):**
- Dense search finds top 10: [Doc_A, Doc_B, Doc_C, ..., Doc_J]
- BM25 reranks ONLY those 10: [Doc_C, Doc_A, Doc_B, ...]
- Final result: Top ranked from the reranked subset

---

#### Summary Table

| Aspect | RRF (Fusion) | Reranking |
|--------|--------------|-----------|
| **Execution** | Parallel | Sequential |
| **Stage 1** | Search full collection | Search full collection |
| **Stage 2** | Search full collection again | Rerank ONLY candidates from stage 1 |
| **Combines** | Complete ranked lists | Filtering + Ranking |
| **Efficiency** | Less efficient (searches full collection twice) | More efficient (reranks small subset) |
| **Use case** | When both methods are equally important | When one method is better at recall, another at precision |

**In your code:**
- `multi_stage_search()` = **Reranking** (prefetch → rerank)
- `rrf_search()` = **RRF/Fusion** (parallel searches → merge)